In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import cv2
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.utils as vutils
from pathlib import Path
from tqdm import tqdm

DRIVE_BASE = "/content/drive/MyDrive"
img_size = (256, 256)
batch_size = 8
num_workers = 2
save_tensors = True
visualize_grid = True
device = "cpu"
output_dir = Path(DRIVE_BASE) / "infosys_internship" / "ASSIGNMENT-01" / "processed_dataset_v2"

(output_dir).mkdir(parents=True, exist_ok=True)
(output_dir / "images").mkdir(exist_ok=True)
(output_dir / "masks").mkdir(exist_ok=True)
(output_dir / "tensors").mkdir(exist_ok=True)
print("Parameters:")
print(f" img_size={img_size}, batch_size={batch_size}, num_workers={num_workers}")
print("Output folders created at:", str(output_dir))

Parameters:
 img_size=(256, 256), batch_size=8, num_workers=2
Output folders created at: /content/drive/MyDrive/infosys_internship/ASSIGNMENT-01/processed_dataset_v2


In [ ]:
def mask_from_otsu(image_bgr):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return th
def mask_from_grabcut(image_bgr, iter_count=5, rect_scale=0.95):
    h, w = image_bgr.shape[:2]
    mask = np.zeros((h, w), np.uint8)
    rect_w = int(w * rect_scale)
    rect_h = int(h * rect_scale)
    rect_x = max(1, (w - rect_w) // 2)
    rect_y = max(1, (h - rect_h) // 2)
    rect = (rect_x, rect_y, rect_w, rect_h)
    bgdModel = np.zeros((1, 65), np.float64)
    fgdModel = np.zeros((1, 65), np.float64)
    try:
        cv2.grabCut(image_bgr, mask, rect, bgdModel, fgdModel, iterCount=iter_count, mode=cv2.GC_INIT_WITH_RECT)
        grabcut_mask = np.where((mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD), 255, 0).astype('uint8')
        if grabcut_mask.sum() < 0.01 * h * w * 255:
            return mask_from_otsu(image_bgr)
        return grabcut_mask
    except Exception as e:
        return mask_from_otsu(image_bgr)
def refine_mask(mask, kernel_size=7, iterations=2):
    if mask is None:
        raise ValueError("refine_mask got None")
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    closed = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=iterations)
    opened = cv2.morphologyEx(closed, cv2.MORPH_OPEN, kernel, iterations=iterations)
    _, binary = cv2.threshold(opened, 127, 255, cv2.THRESH_BINARY)
    return binary


In [ ]:
img_transform = T.Compose([
    T.ToPILImage(),
    T.Resize(img_size, interpolation=Image.BILINEAR),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])
mask_transform = T.Compose([
    T.ToPILImage(),
    T.Resize(img_size, interpolation=Image.NEAREST),
    T.ToTensor(),
])


In [ ]:
class ImageMaskDataset(Dataset):
    def __init__(self, root_dir, generate_masks=True, cache_masks=True):
        self.root_dir = str(root_dir)
        entries = sorted(os.listdir(self.root_dir)) if os.path.isdir(self.root_dir) else []
        self.paths = [os.path.join(self.root_dir, f) for f in entries
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))]
        self.generate_masks = generate_masks
        self.cache_masks = cache_masks
    def __len__(self):
        return len(self.paths)
    def _load_image(self, path):
        img_bgr = cv2.imread(path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise RuntimeError(f"Failed to read image {path}")
        return img_bgr
    def _get_mask_path(self, img_path):
        basename = os.path.splitext(os.path.basename(img_path))[0]
        return os.path.join(output_dir, "masks", f"{basename}_mask.png")
    def _generate_mask(self, img_bgr, img_path):
        mask_path = self._get_mask_path(img_path)
        if self.cache_masks and os.path.exists(mask_path):
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if mask is not None:
                return mask
        mask = mask_from_grabcut(img_bgr)
        mask = refine_mask(mask, kernel_size=7, iterations=2)
        if self.cache_masks:
            cv2.imwrite(mask_path, mask)
        return mask
    def __getitem__(self, idx):
        img_path = self.paths[idx]
        img_bgr = self._load_image(img_path)
        if self.generate_masks:
            mask = self._generate_mask(img_bgr, img_path)
        else:
            mask_file = self._get_mask_path(img_path)
            if os.path.exists(mask_file):
                mask = cv2.imread(mask_file, cv2.IMREAD_GRAYSCALE)
            else:
                mask = mask_from_otsu(img_bgr)
                mask = refine_mask(mask)

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_t = img_transform(img_rgb)
        mask_t = mask_transform(mask)
        mask_t = (mask_t > 0.5).float()
        sample = {
            "img": img_t,
            "mask": mask_t,
            "path": img_path
        }
        return sample


In [ ]:
import torchvision.utils as vutils
import torch
import os

def save_output(image, mask, save_path):
    image = torch.clamp(image, 0.0, 1.0)
    mask = torch.clamp(mask, 0.0, 1.0)

    combined = torch.cat([image, mask.repeat(3, 1, 1)], dim=2)
    vutils.save_image(combined, save_path)


In [ ]:
from tqdm import tqdm

def process_and_save(dataset, batch_size=8, num_workers=2):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    output_base = "/content/drive/MyDrive/infosys_internship/ASSIGNMENT-01/outputs"
    os.makedirs(output_base, exist_ok=True)

    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

    idx = 0
    for batch in tqdm(loader, desc="Batches"):
        imgs = batch["img"]
        masks = batch["mask"]

        imgs = imgs * std + mean
        imgs = torch.clamp(imgs, 0.0, 1.0)

        for i in range(imgs.size(0)):
            save_path = os.path.join(output_base, f"output_{idx}.png")
            save_output(imgs[i], masks[i], save_path)
            idx += 1

    print("Processing finished. Total outputs saved:", idx)


In [ ]:
dataset_dir = "/content/drive/MyDrive/infosys_internship/dataset/val2017/val2017"
print("Using dataset_dir =", dataset_dir)

ds = ImageMaskDataset(
    root_dir=dataset_dir,
    generate_masks=True,
    cache_masks=True
)

print(f"Total images detected: {len(ds)}")

if len(ds) == 0:
    raise SystemExit("No images found. Dataset path is incorrect.")

process_and_save(
    dataset=ds,
    batch_size=batch_size,
    num_workers=num_workers
)

print(" preprocessing completed successfully.")
print("Output saved to:", output_dir)


Using dataset_dir = /content/drive/MyDrive/infosys_internship/dataset/val2017/val2017
Total images detected: 501


Batches: 100%|██████████| 63/63 [36:45<00:00, 35.01s/it]

Processing finished. Total outputs saved: 501
 preprocessing completed successfully.
Output saved to: /content/drive/MyDrive/infosys_internship/ASSIGNMENT-01/processed_dataset_v2


In [ ]:
import os
import cv2

OUTPUT_DIR = "/content/drive/MyDrive/infosys_internship/ASSIGNMENT-01/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def save_output(image, mask, name):
    masked = cv2.bitwise_and(image, image, mask=mask)
    cv2.imwrite(os.path.join(OUTPUT_DIR, name), masked)
